# MASCAF Demo

This notebook walks through the full MASCAF pipeline:

1. Load a demo mesh and bundled skeleton.
2. Visualize the mesh and skeleton.
3. Fit a cable-graph morphology with `CableFitter`.
4. Visualize the resulting SWC model.
5. Run validation and print a summary.

**Requirements:** `mascaf`, `swctools`
(`pip install git+https://github.com/jmrfox/swctools.git`)

In [1]:
import logging
from importlib.resources import files
from pathlib import Path

from mascaf import (
    BasisOptimizerOptions,
    CableFitter,
    FitOptions,
    MeshManager,
    SkeletonGraph,
    Validation,
)
from swctools import SWCModel, plot_model

logging.basicConfig(level=logging.INFO, format="%(levelname)s: %(message)s")

## Choose a demo model

Set `demo_model` to one of `"torus"`, `"cylinder"`, or `"branching"`.

In [2]:
demo_model = "branching"  # "torus" | "cylinder" | "branching"

_DEMO = files("mascaf.demo")
mesh_path = Path(str(_DEMO / f"{demo_model}.obj"))
skeleton_path = Path(str(_DEMO / f"{demo_model}.polylines.txt"))
output_dir = Path("outputs")
output_dir.mkdir(parents=True, exist_ok=True)

print(f"Mesh:     {mesh_path}")
print(f"Skeleton: {skeleton_path}")

Mesh:     C:\Users\MainUser\Documents\Repos\mascaf\mascaf\demo\branching.obj
Skeleton: C:\Users\MainUser\Documents\Repos\mascaf\mascaf\demo\branching.polylines.txt


## Load mesh and skeleton

In [3]:
mm = MeshManager(mesh_path=str(mesh_path))
skeleton = SkeletonGraph.from_txt(str(skeleton_path))

diagonal = mm.bounding_box_diagonal()
print(f"Bounding box diagonal: {diagonal:.4f}")
print(f"Skeleton: {skeleton.number_of_nodes()} nodes")

INFO: Loaded mesh: 1417 vertices, 2830 faces


Bounding box diagonal: 5.7116
Skeleton: 137 nodes


## Visualize mesh

In [4]:
fig = mm.visualize_mesh_3d(skel=None, show_axes=False, title="")
fig.show()

## Visualize mesh with skeleton overlay

In [5]:
fig = mm.visualize_mesh_3d(skel=skeleton, show_axes=False, title="")
fig.show()

## Configure fitting options

`max_edge_length` is set to 10 % of the bounding box diagonal by default.
Adjust `max_edge_length_fraction` or set `max_edge_length` directly.

In [6]:
max_edge_length_fraction = 0.1
max_edge_length = max_edge_length_fraction * diagonal

basis_options = BasisOptimizerOptions(
    do_snapping=True,
    do_forcing=False,
    max_iterations=10,
    lambda_smooth=0.1,
)

fit_options = FitOptions(
    max_edge_length=max_edge_length,
    radius_strategy="equivalent_area",
    basis_optimizer_options=basis_options,
)

print(f"max_edge_length: {max_edge_length:.4f}")

max_edge_length: 0.5712


## Run CableFitter

In [7]:
morphology = CableFitter(fit_options).fit(mm, skeleton)

swc_path = output_dir / f"{demo_model}.swc"
morphology.to_swc_file(str(swc_path))
print(f"SWC written to: {swc_path}")
print(
    f"MorphologyGraph: {morphology.number_of_nodes()} nodes, "
    f"{morphology.number_of_edges()} edges"
)

INFO: Starting cable fit with 137 skeleton nodes, 136 skeleton edges, 1417 mesh vertices, and max_edge_length=0.571164924376903
INFO: Optimizing morphology basis before radius fitting
INFO: Starting basis optimization...
INFO:   Nodes: 15
INFO: Phase 1 - Snapping: 0 nodes outside mesh
INFO: Basis optimization complete
INFO: Finished cable fit with 15 morphology nodes and 14 morphology edges


SWC written to: outputs\branching.swc
MorphologyGraph: 15 nodes, 14 edges


## Visualize morphology

In [10]:
model = SWCModel.from_swc_file(str(swc_path))
model.print_attributes(node_info=False, edge_info=False)

fig = plot_model(
    swc_model=model,
    slider=False,
    title="",
    width=800,
    height=600,
    show_axes=False,
    plot_endcaps=True,
)
fig.show()

INFO: parse_swc start strict=True validate_reconnections=True float_tol=1e-09
INFO: parse_swc done records=15 reconnections=0 header=4
INFO: SWCModel.from_parse_result records=15 reconnections=0 header=4
INFO: SWCModel.from_swc_file built nodes=15 edges=14 strict=True validate_reconnections=True
INFO: batch_frusta count=14 sides=16 end_caps=False verts=448 faces=448
INFO: FrustaSet.from_swc_model edges=14 sides=16 end_caps=False
INFO: plot_model slider=False frusta=14 show_frusta=True show_centroid=True


SWCModel: nodes=15, edges=14, components=1, cycles=0, branch_points=2, roots=1, leaves=3, self_loops=0, density=0.1333


## Validation

In [11]:
validator = Validation(mm, skeleton, morphology)
validator.full_validation()

volume_result = validator.compare_volumes()
area_result = validator.compare_surface_areas()

print(
    f"Volume ratio:      {volume_result['ratio']:.4f} "
    f"(relative error {volume_result['relative_error']:.2%})"
)
print(
    f"Surface area ratio: {area_result['ratio']:.4f} "
    f"(relative error {area_result['relative_error']:.2%})"
)

INFO: Initialized Validation from MorphologyGraph
INFO:   Mesh: 1417 vertices, 2830 faces
INFO:   Skeleton: 137 nodes, 136 edges
INFO:   MorphologyGraph: 15 nodes, 14 edges
INFO: Validation Results, account_for_overlaps=False:
INFO: -- Volume Comparison:
INFO: ---- Mesh volume:       5.1071
INFO: ---- Morphology volume: 3.8494
INFO: ---- Ratio:             0.7537
INFO: ---- Error:             -1.2577
INFO: ---- Relative error:    -24.63%
INFO: -- Surface Area Comparison:
INFO: ---- Mesh area:         22.3177
INFO: ---- Morphology area:   19.5166
INFO: ---- Ratio:             0.8745
INFO: ---- Error:             -2.8011
INFO: ---- Relative error:    -12.55%
INFO: Validation Results, account_for_overlaps=True:
INFO: -- Volume Comparison:
INFO: ---- Mesh volume:       5.1071
INFO: ---- Morphology volume: 2.8803
INFO: ---- Ratio:             0.5640
INFO: ---- Error:             -2.2269
INFO: ---- Relative error:    -43.60%
INFO: -- Surface Area Comparison:
INFO: ---- Mesh area:         22.

Volume ratio:      0.7537 (relative error -24.63%)
Surface area ratio: 0.8745 (relative error -12.55%)
